# Model Definition and Evaluation
## Table of Contents
1. [Model Selection](#model-selection)
2. [Feature Engineering](#feature-engineering)
3. [Hyperparameter Tuning](#hyperparameter-tuning)
4. [Implementation](#implementation)
5. [Evaluation Metrics](#evaluation-metrics)
6. [Comparative Analysis](#comparative-analysis)


In [6]:
#download torch
%pip install torch torchvision

   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
    --------------------------------------- 1.6/123.0 MB 8.4 MB/s eta 0:00:15
    --------------------------------------- 2.9/123.0 MB 6.7 MB/s eta 0:00:18
   - -------------------------------------- 4.2/123.0 MB 6.6 MB/s eta 0:00:18
   - -------------------------------------- 5.8/123.0 MB 6.8 MB/s eta 0:00:18
   -- ------------------------------------- 7.3/123.0 MB 7.0 MB/s eta 0:00:17
   -- ------------------------------------- 8.9/123.0 MB 7.1 MB/s eta 0:00:17
   --- ------------------------------------ 10.7/123.0 MB 7.2 MB/s eta 0:00:16
   ---- ----------------------------------- 12.6/123.0 MB 7.4 MB/s eta 0:00:15
   ---- ----------------------------------- 14.4/123.0 MB 7.5 MB/s eta 0:00:15
   ----- ---------------------------------- 16.5/123.0 MB 7.8 MB/s eta 0:00:14
   ----- ---------------------------------- 18.4/123.0 MB 7.9 MB/s eta 0:00:14
   ------ --------------------------------- 20.4/123.0 MB 8.1 MB/s

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [7]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
# Import models you're considering
import h5py
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import ResNet50_Weights, resnet50


## Model Selection

[Discuss the type(s) of models you consider for this task, and justify the selection.]



We are considering the use of a CNN. We will start with resnet50 as it is a commonly used model for image classification and was used in the literature we found regarding similar work.

In [ ]:
# check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cpu


In [11]:
#utility functions
def denormalize(image_tensor, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return image_tensor * std + mean


def show_original_and_transformed(raw_dataset, transformed_dataset, class_names, indices):
    fig, axes = plt.subplots(len(indices), 2, figsize=(10, 4 * len(indices)))
    if len(indices) == 1:
        axes = [axes]

    for row, idx in enumerate(indices):
        original_image, label = raw_dataset[idx]
        transformed_image, transformed_label = transformed_dataset[idx]

        axes[row][0].imshow(original_image)
        axes[row][0].set_title(f"Original: {class_names[label]}")
        axes[row][0].axis("off")

        transformed_image = denormalize(transformed_image.cpu(), IMAGENET_MEAN, IMAGENET_STD)
        transformed_image = transformed_image.clamp(0, 1).permute(1, 2, 0).numpy()
        axes[row][1].imshow(transformed_image)
        axes[row][1].set_title(f"Transformed: {class_names[transformed_label]}")
        axes[row][1].axis("off")

    plt.tight_layout()
    plt.show()


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def fit_model(model, train_loader, valid_loader, optimizer, criterion, device, epochs):
    history = {
        "train_loss": [],
        "train_acc": [],
        "valid_loss": [],
        "valid_acc": [],
    }

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["valid_loss"].append(valid_loss)
        history["valid_acc"].append(valid_acc)

        print(
            f"Epoch {epoch:02d} | "
            f"train loss: {train_loss:.4f}, train acc: {train_acc:.4f} | "
            f"valid loss: {valid_loss:.4f}, valid acc: {valid_acc:.4f}"
        )

    return history


def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].plot(epochs, history["train_loss"], label="train")
    ax[0].plot(epochs, history["valid_loss"], label="valid")
    ax[0].set_title(f"{title} - Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].set_ylabel("Loss")
    ax[0].legend()

    ax[1].plot(epochs, history["train_acc"], label="train")
    ax[1].plot(epochs, history["valid_acc"], label="valid")
    ax[1].set_title(f"{title} - Accuracy")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylabel("Accuracy")
    ax[1].legend()

    plt.tight_layout()
    plt.show()

In [15]:
# building transformer
#calculate mean and std of the dataset for normalization
def i_sum(X, sample_index):
    return np.sum(X[sample_index,:,:])

num_samples = X.shape[0]

# Calculate intensity sum for each sample and add to labels_df
intensity_sums = [i_sum(X, i) for i in range(num_samples)]


X_mean = np.mean(intensity_sums)
X_mean_1 = np.mean(X)
X_std = np.std(intensity_sums)
X_std_1 = np.std(X)
print(f"Dataset mean: {X_mean} oder {X_mean_1}")
print(f"Dataset std: {X_std} oder {X_std_1}")

# TODO: Build `train_transform` with Resize, CenterCrop, three augmentations of your choice, ToTensor and Normalize
train_transform = transforms.Compose([transforms.ToTensor()])

# TODO: Build `eval_transform` with Resize, CenterCrop, ToTensor and Normalize
eval_transform = transforms.Compose([transforms.ToTensor()])

Dataset mean: 172104.546875 oder 2.76660418510437
Dataset std: 79414.7265625 oder 9.779836654663086


In [ ]:
# implementation of resnet50
num_classes = 5

def initialize_model():
    
    weights = ResNet50_Weights.DEFAULT
    print(weights)
    model = resnet50(weights=weights)

    #  Freeze all parameters
    for param in model.parameters():
        param.requires_grad = False

    #  Replace the classification head with a new layer for `num_classes`
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    # Make the new classification head trainable
    for param in model.fc.parameters():
        param.requires_grad = True

    # Move the model to the correct device
    model = model.to(device)

    return model

model = initialize_model()

print(model.fc)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

ResNet50_Weights.IMAGENET1K_V2
Linear(in_features=2048, out_features=5, bias=True)
Trainable parameters: 10245


## Feature Engineering

[Describe any additional feature engineering you've performed beyond what was done for the baseline model.]


In [4]:
# Load the dataset
# Replace 'your_dataset.csv' with the path to your actual dataset
hf = h5py.File('../../../ML_data/x_y_grid_20260417_105734_020_z000000_snapshot_20260504_103145.h5', 'r')
labels_df = pd.read_csv('../../../ML_data/Classification_table.csv', delimiter=';', usecols=['Symmetrie', 'Resolution'])

# Feature selection
# Example: Selecting only two features for a simple baseline model
#X = hf['entry/acquisition/data']
general_group = hf.get('entry')
acquisition_group = general_group.get('acquisition')
data = acquisition_group.get('data')

# Perform any feature engineering steps
# Example: df['new_feature'] = df['feature1'] + df['feature2']

# we are multiplying the two labels to create a single target variable for classification, where every assymetric spectrum is labeled as 0

# Feature and target variable selection
X = np.transpose(data, (2, 0, 1))   # Samples nach vorne bringen
y = labels_df['Symmetrie'] * labels_df['Resolution']


# Split the dataset into training, validation, and test sets
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15,  random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)  # 0.1765 * 0.85 ≈ 0.15

# verify the splits
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")


Training set size: 1235
Validation set size: 265
Test set size: 265


In [18]:
#dataloader
BATCH_SIZE = 32

valid_loader = DataLoader(X_val, batch_size=BATCH_SIZE, shuffle=False)
train_loader = DataLoader(X_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(X_test, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
#
LEARNING_RATE = 1e-3
NUMBER_OF_EPOCHS = 3
criterion = nn.CrossEntropyLoss()

# Unfreeze the last residual block (`layer4`)
for param in model.layer4.parameters():
    param.requires_grad = True

# TODO: Create a new optimizer for all trainable parameters (use the learning rate defined above)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

# TODO: Re-train the model for a few epochs
history_fine_tune = fit_model(model, train_loader, valid_loader, optimizer, criterion, device, NUMBER_OF_EPOCHS)

# TODO: Plot the second training history
plot_history(history_fine_tune, "Fine Tuned History")

# wir geben im bisher direkt den 3d tensor und ignorieren unsere transformer siehe notebook Woche 5

ValueError: too many values to unpack (expected 2)

## Hyperparameter Tuning

[Discuss any hyperparameter tuning methods you've applied, such as Grid Search or Random Search, and the rationale behind them.]


In [ ]:
# Implement hyperparameter tuning
# Example using GridSearchCV with a DecisionTreeClassifier
# param_grid = {'max_depth': [2, 4, 6, 8]}
# grid_search = GridSearchCV(DecisionTreeClassifier(), param_grid, cv=5)
# grid_search.fit(X_train, y_train)


## Implementation

[Implement the final model(s) you've selected based on the above steps.]


In [ ]:
# Implement the final model(s)
# Example: model = YourChosenModel(best_hyperparameters)
# model.fit(X_train, y_train)


## Evaluation Metrics

[Clearly specify which metrics you'll use to evaluate the model performance, and why you've chosen these metrics.]


In [ ]:
# Evaluate the model using your chosen metrics
# Example for classification
# y_pred = model.predict(X_test)
# print(classification_report(y_test, y_pred))

# Example for regression
# mse = mean_squared_error(y_test, y_pred)

# Your evaluation code here


## Comparative Analysis

[Compare the performance of your model(s) against the baseline model. Discuss any improvements or setbacks and the reasons behind them.]


In [ ]:
# Comparative Analysis code (if applicable)
# Example: comparing accuracy of the baseline model and the new model
# print(f"Baseline Model Accuracy: {baseline_accuracy}, New Model Accuracy: {new_model_accuracy}")
